# STFPM 분할 비교 — 5장 간격 분할 vs 시간 블록 (같은 코드·같은 설정)

**왜 하나**
- 같은 새 구조(layer2+3, 20 epoch, mean)인데 팀원 보고서는 AUROC 0.972, 우리 시간 블록 재현은 0.788(공통 사진, fold 평균)이다.
- 가장 큰 차이는 **정상 사진 분할**이다. 팀원 방식은 정상 2,000장 중 5장마다 1장을 시험으로 뺀다 → 시험 정상마다 바로 옆 프레임(1장 차이)이 학습에 들어 있다.
- 이번엔 **분할만 팀원 방식으로 바꾸고 나머지는 시간 블록 실행과 똑같이** 둔다. 0.97 근처가 나오면 두 숫자의 차이는 분할 때문이라고 확인된다.

| 항목 | 시간 블록 실행 (10/14, 이미 있음) | 이번 실행 |
|---|---|---|
| 정상 분할 | 시간 블록 5-fold + buffer 30 | **5장 간격 1회** (학습 후보 1,600 / 시험 400) |
| 학습 정상 | 학습 구간 ∩ 2단계 v3.2 | 학습 후보 ∩ 2단계 v3.2 (1,216장) |
| 시험 | fold별 검증 구간 | 시험 정상 400 + 언더필 2,000 |
| 코드·설정·입력 | 팀원 v2 코드, layer2+3, 20 epoch, mean, 기준 95%, 원본 크롭 PNG | **같음** |

- 5장 중 몇 번째를 뺄지(OFFSET)는 팀원 README에 없어서 0번째로 정했다. 어느 걸 빼도 "옆 프레임이 학습에 있다"는 구조는 같다.
- 팀원 선별(현환님 초기 2단계, 통과 61%)은 목록이 없어서 맞추지 못했다. 우리 2단계 v3.2와 옛 2단계(고정 크롭)로 나눠 보고한다.

**Drive `산공종설/`에 필요한 파일**
| 파일 | 상태 |
|---|---|
| underfill_originals_4000.zip | 이미 있음 |
| stfpm_v2_timeblock_mask_bundle_20261006.zip | 이미 있음 |
| dinov2_grid_bundle_20261013.zip | 이미 있음 |
| results/stfpm_stage2v32_20261013/ (시간 블록 점수, 비교용) | 이미 있음 |
| **stfpm_stride5_bundle_20261014.zip** | **새로 올림** |

**시간 [추정]:** 설치 5분 + 입력 준비 20~40분(원본 크롭 1벌, 저장 압축을 낮춰 지난번보다 빠름) + 학습·점수 1회 40분~1시간(L4). T4면 더 걸린다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인
시간 블록 점수 폴더가 `OK`여야 8번 셀에서 같은 사진 비교가 된다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/stfpm_stride5_20261014"
TIMEBLOCK = f"{DRIVE_DIR}/results/stfpm_stage2v32_20261013/s2_raw_mean"
os.makedirs(RESULTS, exist_ok=True)
FILES = ("underfill_originals_4000.zip", "stfpm_v2_timeblock_mask_bundle_20261006.zip", "dinov2_grid_bundle_20261013.zip",
         "stfpm_stride5_bundle_20261014.zip")
for f in FILES:
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)
print("OK  " if all(os.path.exists(f"{TIMEBLOCK}/fold{k}/scores.csv") for k in range(5)) else "없음", "시간 블록 점수", TIMEBLOCK)

## 3. 압축 해제
마지막 줄에 원본 `4000`이 나와야 한다.

In [ ]:
for f in FILES:
    shutil.copy2(f"{DRIVE_DIR}/{f}", "/content/")
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!rm -rf /content/stfpm_v2 /content/grid /content/st5
!unzip -q -o /content/stfpm_v2_timeblock_mask_bundle_20261006.zip -d /content/stfpm_v2
!unzip -q -o /content/dinov2_grid_bundle_20261013.zip -d /content/grid
!unzip -q -o /content/stfpm_stride5_bundle_20261014.zip -d /content/st5
!find /content/orig -name "*.jpg" | wc -l

## 4. 패키지 설치
지난번 STFPM 실행과 같은 버전(anomalib 2.6.2).

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md
print({p: md.version(p) for p in ("anomalib", "torch", "lightning")})
import timm
timm.create_model("resnet18", pretrained=True)   # 팀원 train.py는 HF_HUB_OFFLINE=1로 돌아서, 사전학습 가중치를 미리 받아 둔다
print("resnet18 가중치 준비 완료")

## 5. 입력 준비 (원본 크롭 PNG 4,000장 + 분할 목록)
`분할:` 줄에서 확인할 것: `test_normal 400`, `train_stage2 1216`, `val_total 2400`, `max_gap_to_train 1`.
- `max_gap_to_train 1` = 모든 시험 정상 사진의 바로 옆 프레임이 학습 후보에 있다는 뜻 (시간 블록은 최소 31프레임 떨어짐). 이번 비교의 핵심 차이다.

In [ ]:
!python /content/st5/prepare_stfpm_stride5.py --self-test --grid /content/grid
!python /content/st5/prepare_stfpm_stride5.py --grid /content/grid --orig /content/orig --out /content/stfpm_inputs

## 6. 학습 + 점수 (1회)
래퍼와 팀원 코드는 시간 블록 실행과 같은 파일이다. 로그는 Drive `results/stfpm_stride5_20261014/stride5_raw_mean/fold0/train_log.txt`.

In [ ]:
!python /content/stfpm_v2/stfpm_timeblock.py --project /content/stfpm_v2/STFPM_Defect_Detection --orig-root /content/stfpm_inputs --no-roi-crop --split-csv /content/stfpm_inputs/split_stride5.csv --path-column raw_crop --model-config /content/st5/configs/stfpm_l23_mean.yaml --work /content/stfpm_work --train-args '--percentile 0.95' --out "{RESULTS}/stride5_raw_mean"

## 7. 결과: 같은 시험 사진에서 두 분할 비교
- **5장 간격:** 모델 1개라 AUROC를 한 번에 계산한다 (팀원 보고서와 같은 방식).
- **시간 블록:** 같은 사진들이 서로 다른 fold 모델로 점수를 받았다. 그래서 fold별 AUROC 평균을 주 비교로 쓰고, 한꺼번에 계산한 값은 참고로만 둔다(fold마다 점수 크기가 달라 섞이면 왜곡된다).
- 묶음: 시험 2,400장 전부 / 2단계 v3.2 통과 / 옛 2단계(고정 크롭) 통과.
- **판정:** 5장 간격 AUROC가 0.97 근처이고 시간 블록이 훨씬 낮으면 → 차이는 분할 때문.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
sel = pd.read_csv("/content/grid/selections.csv", encoding="utf-8-sig").set_index("source")
st5 = pd.read_csv(f"{RESULTS}/stride5_raw_mean/fold0/scores.csv", encoding="utf-8-sig")
tb = pd.concat([pd.read_csv(f"{TIMEBLOCK}/fold{k}/scores.csv", encoding="utf-8-sig") for k in range(5)])
for d in (st5, tb):
    d["source"] = d.source.str.replace(chr(92), "/")
tb = tb.set_index("source").loc[st5.source].reset_index()     # 같은 시험 사진만
assert len(st5) == len(tb) == 2400
groups = {"시험 2,400 전부": st5.source.map(lambda s: True),
          "2단계 v3.2 통과": st5.source.map(sel.stage2) == 1,
          "옛 2단계(고정 크롭) 통과": st5.source.map(sel.clean_old) == 1}
def rates(d):
    y, p = d.label_id, d.pred
    return int(((p == 1) & (y == 0)).sum()), int(((p == 0) & (y == 1)).sum()), round(((p == 1) & (y == 0)).sum() / (y == 0).sum(), 4), round(((p == 1) & (y == 1)).sum() / (y == 1).sum(), 4)
rows = []
for name, m in groups.items():
    a, b = st5[m.values], tb[m.values]
    fold_auc = [roc_auc_score(g.label_id, g.score) for _, g in b.groupby("fold") if g.label_id.nunique() == 2]
    fp, fn, fpr, rec = rates(a); fp2, fn2, fpr2, rec2 = rates(b)
    rows.append({"묶음": name, "n": int(m.sum()), "정상": int((a.label_id == 0).sum()),
                 "5장간격 AUROC": round(roc_auc_score(a.label_id, a.score), 4),
                 "시간블록 fold평균 AUROC": round(np.mean(fold_auc), 4), "(참고) 시간블록 한꺼번 AUROC": round(roc_auc_score(b.label_id, b.score), 4),
                 "5장간격 오탐률/재현율": f"{fpr}/{rec}", "시간블록 오탐률/재현율": f"{fpr2}/{rec2}"})
table = pd.DataFrame(rows); table.to_csv(f"{RESULTS}/split_compare.csv", index=False, encoding="utf-8-sig")
print(table.to_string(index=False))

## 8. 결과 묶기
점수·로그·설정만 Drive `results/stfpm_stride5_20261014_scores.zip`으로 묶는다(체크포인트 제외).

In [ ]:
import zipfile, glob
ZIP = f"{DRIVE_DIR}/results/stfpm_stride5_20261014_scores.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt", ".yaml")):
            z.write(p, os.path.relpath(p, RESULTS))
    z.write("/content/stfpm_inputs/split_stride5.csv", "split_stride5.csv")
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 9. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()